# MIMIC-IV EDA — Lung Cancer Cohort Identification

This notebook performs exploratory data analysis on MIMIC-IV to identify and characterise the lung cancer
patient cohort (ICD-10 codes C34.0–C34.9, "Malignant neoplasm of bronchus and lung"). It loads structured
EHR tables from **MIMIC-IV 3.1 (hosp module)** — `patients`, `admissions`, `diagnoses_icd`, and
`d_icd_diagnoses` — filters them to lung cancer diagnoses, and cross-references the resulting cohort against
**MIMIC-IV-Note 2.2** (`discharge` and `radiology` notes) to assess clinical text availability. The expected
outputs are: a characterised cohort (demographics, ICD subcode distribution, note availability) and three
saved CSVs (`cohort_patients.csv`, `cohort_admissions.csv`, `cohort_notes_index.csv`) for downstream use.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)

In [ ]:
DATA_ROOT = Path('../../data/raw')
MIMIC_HOSP = DATA_ROOT / 'mimic-iv-3.1/hosp'
MIMIC_NOTE = DATA_ROOT / 'mimic-iv-note-2.2/note'

for name, path in [('DATA_ROOT', DATA_ROOT), ('MIMIC_HOSP', MIMIC_HOSP), ('MIMIC_NOTE', MIMIC_NOTE)]:
    print(f"{name}: {path} -> exists: {path.exists()}")

## Load Structured MIMIC-IV Tables

In [ ]:
patients = pd.read_csv(MIMIC_HOSP / 'patients.csv.gz', compression='gzip')
admissions = pd.read_csv(MIMIC_HOSP / 'admissions.csv.gz', compression='gzip')
diagnoses = pd.read_csv(MIMIC_HOSP / 'diagnoses_icd.csv.gz', compression='gzip')
d_icd = pd.read_csv(MIMIC_HOSP / 'd_icd_diagnoses.csv.gz', compression='gzip')

for name, df in [('patients', patients), ('admissions', admissions),
                 ('diagnoses', diagnoses), ('d_icd', d_icd)]:
    print(f"--- {name} ---")
    print(f"shape: {df.shape}")
    print(f"memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    display(df.head())
    print()

## Filter to Lung Cancer Cohort (ICD-10 C34.x)

ICD-10 code **C34** denotes "Malignant neoplasm of bronchus and lung". Subcodes **C34.0–C34.9** specify the
anatomical site (e.g. main bronchus, upper/middle/lower lobe, overlapping sites, unspecified). We filter the
diagnoses table on this entire code family (`C34*`) to capture all lung cancer diagnoses regardless of
subsite, since the NSCLC TNM staging project is concerned with the disease as a whole.

In [ ]:
c34_codes = d_icd[(d_icd['icd_version'] == 10) & (d_icd['icd_code'].str.startswith('C34'))]
display(c34_codes)

In [ ]:
c34_code_set = set(c34_codes['icd_code'])

lung_cancer_diagnoses = diagnoses[diagnoses['icd_code'].isin(c34_code_set)]

print(f"Total diagnosis records: {len(lung_cancer_diagnoses)}")
print(f"Unique patients (subject_id): {lung_cancer_diagnoses['subject_id'].nunique()}")
print(f"Unique admissions (hadm_id): {lung_cancer_diagnoses['hadm_id'].nunique()}")

In [ ]:
cohort = (
    lung_cancer_diagnoses
    .merge(patients, on='subject_id', how='left')
    .merge(admissions, on=['subject_id', 'hadm_id'], how='left')
)

print(f"shape: {cohort.shape}")
display(cohort.head())

## Cohort Characterisation

In [ ]:
# Demographics are per-patient attributes; deduplicate by subject_id so patients
# with multiple lung-cancer admissions/codes aren't over-weighted in the plots.
cohort_unique = cohort.drop_duplicates(subset='subject_id')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Age distribution
axes[0, 0].hist(cohort_unique['anchor_age'], bins=20, edgecolor='black')
axes[0, 0].set_title('Age Distribution (anchor_age)')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Number of patients')

# Gender distribution
sns.countplot(data=cohort_unique, x='gender', ax=axes[0, 1])
axes[0, 1].set_title('Gender Distribution')

# Anchor year group distribution
order = cohort_unique['anchor_year_group'].value_counts().index
sns.countplot(data=cohort_unique, y='anchor_year_group', order=order, ax=axes[1, 0])
axes[1, 0].set_title('Anchor Year Group Distribution')

# Summary stats
axes[1, 1].axis('off')
age_mean = cohort_unique['anchor_age'].mean()
age_median = cohort_unique['anchor_age'].median()
age_std = cohort_unique['anchor_age'].std()
stats_text = (
    f"Cohort summary (unique patients)\n\n"
    f"N patients: {len(cohort_unique)}\n\n"
    f"Age mean:   {age_mean:.1f}\n"
    f"Age median: {age_median:.1f}\n"
    f"Age std:    {age_std:.1f}"
)
axes[1, 1].text(0.05, 0.5, stats_text, fontsize=12, va='center', family='monospace')

plt.tight_layout()
plt.show()

print(f"Age (unique patients) — mean: {age_mean:.2f}, median: {age_median:.2f}, std: {age_std:.2f}")

In [ ]:
icd_counts = (
    lung_cancer_diagnoses['icd_code']
    .value_counts()
    .rename_axis('icd_code')
    .reset_index(name='count')
    .merge(c34_codes[['icd_code', 'long_title']], on='icd_code', how='left')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=icd_counts, x='count', y='icd_code', ax=ax, order=icd_counts['icd_code'])

for i, row in icd_counts.iterrows():
    ax.text(row['count'], i, f"  {row['long_title']}", va='center', fontsize=8)

ax.set_title('Distribution of C34.x ICD-10 Codes in Lung Cancer Cohort')
ax.set_xlabel('Number of diagnosis records')
ax.set_ylabel('ICD-10 code')
plt.tight_layout()
plt.show()

icd_counts

## Cross-reference with MIMIC-IV-Note

In [ ]:
# discharge.csv.gz (~1.1 GB) and radiology.csv.gz (~750 MB) are dominated by the free-text 'text'
# column. We don't need note text for cohort counting, so we exclude it via usecols to keep
# memory usage manageable; the Clinical Context Agent will load text per-patient on demand.
note_cols = ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime']

discharge_notes = pd.read_csv(MIMIC_NOTE / 'discharge.csv.gz', compression='gzip', usecols=note_cols)
radiology_notes = pd.read_csv(MIMIC_NOTE / 'radiology.csv.gz', compression='gzip', usecols=note_cols)

for name, df in [('discharge_notes', discharge_notes), ('radiology_notes', radiology_notes)]:
    print(f"--- {name} ---")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print()

In [ ]:
cohort_subject_ids = cohort['subject_id'].unique()

discharge_counts = (
    discharge_notes[discharge_notes['subject_id'].isin(cohort_subject_ids)]
    .groupby('subject_id').size().rename('n_discharge_notes')
)
radiology_counts = (
    radiology_notes[radiology_notes['subject_id'].isin(cohort_subject_ids)]
    .groupby('subject_id').size().rename('n_radiology_notes')
)

note_counts = (
    pd.DataFrame(index=pd.Index(cohort_subject_ids, name='subject_id'))
    .join(discharge_counts)
    .join(radiology_counts)
    .fillna(0)
    .astype(int)
)

n_cohort = len(note_counts)
n_with_discharge = (note_counts['n_discharge_notes'] > 0).sum()
n_with_radiology = (note_counts['n_radiology_notes'] > 0).sum()
n_with_both = ((note_counts['n_discharge_notes'] > 0) & (note_counts['n_radiology_notes'] > 0)).sum()

print(f"Cohort size (unique patients): {n_cohort}")
print(f"Patients with >=1 discharge note: {n_with_discharge}")
print(f"Patients with >=1 radiology note: {n_with_radiology}")
print(f"Patients with both: {n_with_both}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(note_counts['n_discharge_notes'], bins=20, edgecolor='black')
axes[0].set_title('Discharge Notes per Patient')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Number of patients')

axes[1].hist(note_counts['n_radiology_notes'], bins=20, edgecolor='black')
axes[1].set_title('Radiology Notes per Patient')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Number of patients')

plt.tight_layout()
plt.show()

## Save Cohort for Downstream Use

In [ ]:
OUT_DIR = Path('../../data/processed/cohort')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# cohort_patients.csv: subject_ids only
cohort_patients = pd.DataFrame({'subject_id': cohort['subject_id'].unique()})
cohort_patients.to_csv(OUT_DIR / 'cohort_patients.csv', index=False)

# cohort_admissions.csv: full per-admission table (one row per diagnosis record)
cohort.to_csv(OUT_DIR / 'cohort_admissions.csv', index=False)

# cohort_notes_index.csv: subject_id -> list of note_ids (discharge + radiology)
notes_combined = pd.concat([
    discharge_notes.loc[discharge_notes['subject_id'].isin(cohort_patients['subject_id']), ['subject_id', 'note_id']],
    radiology_notes.loc[radiology_notes['subject_id'].isin(cohort_patients['subject_id']), ['subject_id', 'note_id']],
])
cohort_notes_index = (
    notes_combined.groupby('subject_id')['note_id']
    .apply(list)
    .reset_index()
    .rename(columns={'note_id': 'note_ids'})
)
cohort_notes_index.to_csv(OUT_DIR / 'cohort_notes_index.csv', index=False)

print(f"cohort_patients.csv:     {len(cohort_patients)} rows")
print(f"cohort_admissions.csv:   {len(cohort)} rows")
print(f"cohort_notes_index.csv:  {len(cohort_notes_index)} rows")

## Summary of Key Findings

**Cohort identification (ICD-10 C34.x — malignant neoplasm of bronchus and lung)**
- **2,283 unique patients** identified across **4,270 diagnosis records** spanning **4,145 unique admissions**
- 16 distinct C34.x subcodes present; the most frequent are:
  - `C3490` — unspecified part of unspecified bronchus or lung (984 records)
  - `C3411` — upper lobe, right bronchus or lung (887 records)
  - `C3412` — upper lobe, left bronchus or lung (686 records)
  - `C3431` / `C3432` — lower lobe, right/left bronchus or lung (485 / 374 records)

**Demographics (per unique patient)**
- Age: mean **68.0**, median **68.0**, std **11.0** (range roughly 25–91, concentrated 55–85)
- Gender: **53.2% female (1,215)**, **46.8% male (1,068)**
- Anchor year groups skew toward more recent years (2017–2019 is the largest group), consistent with MIMIC-IV's later cohort coverage

**Clinical note availability (MIMIC-IV-Note 2.2)**
- Of 2,283 cohort patients:
  - **1,563 (68.5%)** have ≥1 discharge summary
  - **1,591 (69.7%)** have ≥1 radiology report
  - **1,527 (66.9%)** have both note types
  - **1,627 (71.3%)** have at least one note of either type
- Discharge notes per patient are mostly low-count (1–5), while radiology notes are more numerous per patient (often 10–50+), reflecting repeated imaging during workup/staging

**Outputs saved to `data/processed/cohort/`**
- `cohort_patients.csv` — 2,283 rows (unique `subject_id`)
- `cohort_admissions.csv` — 4,270 rows (one row per lung-cancer diagnosis record, joined with patient/admission attributes)
- `cohort_notes_index.csv` — 1,627 rows (`subject_id` → list of discharge + radiology `note_id`s)

**Implications for the Clinical Context Agent**
- ~71% of the cohort has at least some clinical text available, but ~29% (656 patients) have no notes in MIMIC-IV-Note and will need to rely solely on structured data (or be excluded from the RAG pipeline)
- Radiology reports are the dominant text source by volume and are likely to contain TNM-relevant descriptors (tumor size, nodal involvement, metastases) most directly